# 🧩 Notebook 3 · Restaurant — Polymorphism & Patterns

Notebook 2 gave us a working POS. Now we make it *extensible* using three
design patterns you'll meet in every OOD interview:

1. **Strategy** — swap pricing/discount rules (standard, happy-hour, loyalty)
   without touching `Order` or `Bill`.
2. **Factory** — build `Menu` and `MenuItem` objects from configuration
   (JSON / a dict) so non-programmers can edit the menu.
3. **Observer** — when an order is *placed* or *served*, notify the kitchen
   screen, an SMS to the guest, and the manager's dashboard — all decoupled.

> **Why patterns?** They're *names* for shapes you'd invent anyway. Knowing
> the name helps two developers agree on an idea in one word.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/restaurant
uv sync
```

Select the `.venv` kernel. If missing → `Cmd+Shift+P` → **Reload Window**.


## 1️⃣ Shared domain (copied from Notebook 2 so this runs alone)


In [1]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import time
from enum import Enum
from itertools import count


@dataclass(frozen=True)
class MenuItem:
    name: str
    price: float
    category: str
    vegetarian: bool = False


class OrderStatus(Enum):
    OPEN = "open"; PLACED = "placed"; SERVED = "served"; PAID = "paid"


class TableState(Enum):
    FREE = "free"; TAKEN = "taken"; RESERVED = "reserved"


@dataclass
class Table:
    number: int
    seats: int
    state: TableState = TableState.FREE
    def free(self): self.state = TableState.FREE


@dataclass
class OrderItem:
    item: MenuItem
    qty: int = 1
    def line_total(self) -> float: return round(self.item.price * self.qty, 2)


_oids = count(1)

@dataclass
class Order:
    table: Table
    id: int = field(default_factory=lambda: next(_oids))
    items: list[OrderItem] = field(default_factory=list)
    status: OrderStatus = OrderStatus.OPEN
    def add(self, item, qty=1): self.items.append(OrderItem(item, qty))
    def subtotal(self) -> float: return round(sum(i.line_total() for i in self.items), 2)


## 2️⃣ Strategy — swap pricing rules at runtime

### ❌ Bad: hard-coded tax + tip in `Order.pay()`

```python
def pay(self):
    sub = self.subtotal()
    return sub + sub*0.08 + sub*0.15  # magic numbers, one formula forever
```

If marketing runs a "20% off pizzas on Tuesdays" promo, you're editing
`Order.pay()` again. Every promo = another `if`. Ten promos later the method
is unreadable.

### ✅ Best: a `PricingStrategy` interface

The `Order` depends on an **abstract** strategy. You can plug in any rule
without changing the order code. (Open/Closed principle.)


In [2]:
class PricingStrategy(ABC):
    """Given an order, return the final bill as a dict."""

    @abstractmethod
    def bill(self, order: Order) -> dict: ...


class StandardPricing(PricingStrategy):
    """Subtotal + 8% tax + 15% tip."""

    def __init__(self, tax_rate=0.08, tip_rate=0.15):
        self.tax_rate, self.tip_rate = tax_rate, tip_rate

    def bill(self, order: Order) -> dict:
        sub = order.subtotal()
        tax = round(sub * self.tax_rate, 2)
        tip = round(sub * self.tip_rate, 2)
        return {"subtotal": sub, "tax": tax, "tip": tip,
                "total": round(sub + tax + tip, 2)}


class HappyHourPricing(PricingStrategy):
    """20% off drinks between 17:00 and 19:00, same tax/tip otherwise."""

    def __init__(self, now: time, base: PricingStrategy):
        self.now, self.base = now, base

    def bill(self, order: Order) -> dict:
        discount = 0.0
        if time(17, 0) <= self.now < time(19, 0):
            discount = sum(i.line_total() for i in order.items
                           if i.item.category == "drinks") * 0.20
        result = self.base.bill(order)
        result["discount"] = round(discount, 2)
        result["total"] = round(result["total"] - discount, 2)
        return result


class LoyaltyPricing(PricingStrategy):
    """Loyalty members get 10% off the subtotal (before tax/tip).

    This class is a *decorator* over another strategy: it tweaks the bill
    produced by ``base`` without caring how the base computed it.
    """

    def __init__(self, base: PricingStrategy, member: bool):
        self.base, self.member = base, member

    def bill(self, order: Order) -> dict:
        result = self.base.bill(order)
        if not self.member:
            return result
        saved = round(result["subtotal"] * 0.10, 2)
        result["loyalty_discount"] = saved
        result["total"] = round(result["total"] - saved, 2)
        return result


# --- demo -------------------------------------------------------------------
menu_items = {
    "Margherita": MenuItem("Margherita", 10, "pizza", True),
    "Coke":       MenuItem("Coke",        3, "drinks", True),
}
t = Table(1, 2, TableState.TAKEN)
o = Order(table=t)
o.add(menu_items["Margherita"])
o.add(menu_items["Coke"], qty=2)

standard  = StandardPricing()
happy     = HappyHourPricing(now=time(18, 0), base=standard)
loyal     = LoyaltyPricing(base=standard, member=True)

print("standard  :", standard.bill(o))
print("happy hour:", happy.bill(o))
print("loyalty   :", loyal.bill(o))


standard  : {'subtotal': 16, 'tax': 1.28, 'tip': 2.4, 'total': 19.68}
happy hour: {'subtotal': 16, 'tax': 1.28, 'tip': 2.4, 'total': 18.48, 'discount': 1.2}
loyalty   : {'subtotal': 16, 'tax': 1.28, 'tip': 2.4, 'total': 18.08, 'loyalty_discount': 1.6}


💡 **Real-world parallel:** payment processors (Stripe, PayPal) use the
same pattern — your checkout code talks to a `PaymentStrategy`, and you
swap providers without touching the cart.


## 3️⃣ Factory — build menus from config

### ❌ Bad: new items are hard-coded in Python

```python
menu = Menu([
    MenuItem("Margherita", 10, "pizza"),
    MenuItem("Pepperoni", 12, "pizza"),
    ...  # change requires a developer + deploy
])
```

Restaurants change menus weekly. Non-developers (the manager) should be
able to edit a JSON file.

### ✅ Best: a `MenuFactory` that reads config


In [3]:
import json

SAMPLE_CONFIG = """
[
    {"name": "Margherita", "price": 10.0, "category": "pizza", "vegetarian": true},
    {"name": "Pepperoni",  "price": 12.0, "category": "pizza"},
    {"name": "Caesar",     "price":  8.0, "category": "salad",   "vegetarian": true},
    {"name": "Coke",       "price":  3.0, "category": "drinks",  "vegetarian": true},
    {"name": "Tiramisu",   "price":  6.0, "category": "dessert", "vegetarian": true}
]
"""


class MenuFactory:
    """Build MenuItems from a dict/JSON. One place knows the schema."""

    @staticmethod
    def from_dict(d: dict) -> MenuItem:
        required = {"name", "price", "category"}
        missing = required - d.keys()
        if missing:
            raise ValueError(f"menu item missing fields: {missing}")
        return MenuItem(
            name=d["name"],
            price=float(d["price"]),
            category=d["category"],
            vegetarian=bool(d.get("vegetarian", False)),
        )

    @classmethod
    def from_json(cls, text: str) -> list[MenuItem]:
        return [cls.from_dict(d) for d in json.loads(text)]


items = MenuFactory.from_json(SAMPLE_CONFIG)
for mi in items:
    tag = " 🌱" if mi.vegetarian else ""
    print(f"{mi.name:<12} ${mi.price:5.2f}  [{mi.category}]{tag}")

assert any(i.vegetarian for i in items)


Margherita   $10.00  [pizza] 🌱
Pepperoni    $12.00  [pizza]
Caesar       $ 8.00  [salad] 🌱
Coke         $ 3.00  [drinks] 🌱
Tiramisu     $ 6.00  [dessert] 🌱


💡 **Real-world parallel:** web frameworks like Django use factories to
turn database rows into model instances, and game engines use them to spawn
enemies from level files.


## 4️⃣ Observer — notify many systems when something happens

When an order is *placed*:

- the **kitchen display** should show it,
- the **guest** should get an SMS ("your order is in!"),
- the **manager dashboard** should update "open orders" counter.

### ❌ Bad: `Order.place()` imports all three and calls them

Tight coupling. Changing the SMS provider changes `Order`. Testing `Order`
requires mocking three modules.

### ✅ Best: `Order` emits events; subscribers register themselves


In [4]:
class OrderEvents:
    """Tiny publish/subscribe hub keyed by event name."""

    def __init__(self):
        self._subs: dict[str, list] = {}

    def subscribe(self, event: str, fn) -> None:
        self._subs.setdefault(event, []).append(fn)

    def publish(self, event: str, order: Order) -> None:
        for fn in self._subs.get(event, []):
            fn(order)


class ObservableOrder(Order):
    """Same as Order, but broadcasts lifecycle events."""

    events: OrderEvents = None  # shared class-level hub (for simplicity)

    def place(self):
        if self.status != OrderStatus.OPEN:
            raise ValueError("order already placed")
        if not self.items:
            raise ValueError("empty order")
        self.status = OrderStatus.PLACED
        self.events.publish("placed", self)

    def serve(self):
        if self.status != OrderStatus.PLACED:
            raise ValueError("order not placed")
        self.status = OrderStatus.SERVED
        self.events.publish("served", self)


# --- subscribers ------------------------------------------------------------
def kitchen_display(order: Order) -> None:
    names = ", ".join(f"{oi.qty}x {oi.item.name}" for oi in order.items)
    print(f"[KITCHEN]  order {order.id} → {names}")


def sms_to_guest(order: Order) -> None:
    print(f"[SMS]      order {order.id}: your food is on the way 🍕")


def manager_dashboard(order: Order) -> None:
    print(f"[DASH]     table {order.table.number} event: {order.status.value}")


# --- wiring -----------------------------------------------------------------
hub = OrderEvents()
hub.subscribe("placed", kitchen_display)
hub.subscribe("placed", sms_to_guest)
hub.subscribe("placed", manager_dashboard)
hub.subscribe("served", manager_dashboard)

ObservableOrder.events = hub

t2 = Table(2, 4, TableState.TAKEN)
o2 = ObservableOrder(table=t2)
o2.add(menu_items["Margherita"])
o2.add(menu_items["Coke"])
o2.place()
o2.serve()


[KITCHEN]  order 2 → 1x Margherita, 1x Coke
[SMS]      order 2: your food is on the way 🍕
[DASH]     table 2 event: placed
[DASH]     table 2 event: served


💡 **Real-world parallel:** every modern chat/social app works this way.
When you post a message, the backend *publishes* a `message_created` event;
separate services handle notifications, feed updates, analytics, and search
indexing — each one added without touching the others.


## 5️⃣ All three patterns together

In production you'd combine them: the **Factory** builds menus from config,
the **Strategy** calculates the bill, and the **Observer** fans events out to
downstream systems. Each pattern protects you from a different kind of
change.

| Change that happens a lot | Pattern that absorbs it |
|---|---|
| New discount / tax rules | **Strategy** |
| New menu items daily | **Factory** |
| New downstream system (Slack alert, loyalty CRM) | **Observer** |

## 🎓 Interview-style recap

If an interviewer asks *"how would you design a restaurant POS?"* — walk
them through:

1. **Domain model** (Notebook 1): Menu, Table, Order, Bill, Staff.
2. **State machines** (Notebook 2) to rule out invalid transitions.
3. **Patterns** (this notebook) for the parts that change most.

That's a 20-minute answer that shows you think about *evolution*, not just
one-shot code.

## 🔭 Further practice

- Add a **Decorator** to `MenuItem` for "extra cheese / gluten-free" that
  bumps the price.
- Use the **State pattern** to replace the `_require` checks in `Order`
  with one class per status.
- Add a **Repository** (SQLite) that persists orders and bills.
